In [1]:
import pandas as pd

df = pd.read_csv('results.csv')

print(df.shape)
print(df.columns.tolist())
df.head(10)

(49287, 9)
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False
5,1876-03-25,Scotland,Wales,4.0,0.0,Friendly,Glasgow,Scotland,False
6,1877-03-03,England,Scotland,1.0,3.0,Friendly,London,England,False
7,1877-03-05,Wales,Scotland,0.0,2.0,Friendly,Wrexham,Wales,False
8,1878-03-02,Scotland,England,7.0,2.0,Friendly,Glasgow,Scotland,False
9,1878-03-23,Scotland,Wales,9.0,0.0,Friendly,Glasgow,Scotland,False


In [2]:
def get_result(row):
    if row['home_score'] > row['away_score']:
        return 'home_win'
    elif row['home_score'] < row['away_score']:
        return 'away_win'
    else:
        return 'draw'

df['result'] = df.apply(get_result, axis=1)

print(df['result'].value_counts())

result
home_win    24106
away_win    13912
draw        11269
Name: count, dtype: int64


In [3]:
# Check if Nepal exists in the dataset
nepal_home = df[df['home_team'] == 'Nepal']
nepal_away = df[df['away_team'] == 'Nepal']

print("Nepal home matches:", len(nepal_home))
print("Nepal away matches:", len(nepal_away))

Nepal home matches: 126
Nepal away matches: 142


In [4]:
def get_team_stats(team_name, data):
    home_matches = data[data['home_team'] == team_name]
    away_matches = data[data['away_team'] == team_name]
    total_matches = len(home_matches) + len(away_matches)
    home_wins = len(home_matches[home_matches['result'] == 'home_win'])
    away_wins = len(away_matches[away_matches['result'] == 'away_win'])
    total_wins = home_wins + away_wins
    goals_scored = home_matches['home_score'].sum() + away_matches['away_score'].sum()
    goals_conceded = home_matches['away_score'].sum() + away_matches['home_score'].sum()
    win_rate = total_wins / total_matches if total_matches > 0 else 0
    avg_goals_scored = goals_scored / total_matches if total_matches > 0 else 0
    avg_goals_conceded = goals_conceded / total_matches if total_matches > 0 else 0

    print("Total Matches:", total_matches)
    print("Win Rate:", round(win_rate, 3))
    print("Avg Goals Scored:", round(avg_goals_scored, 3))
    print("Avg Goals Conceded:", round(avg_goals_conceded, 3))

get_team_stats('Nepal', df)

Total Matches: 268
Win Rate: 0.235
Avg Goals Scored: 0.854
Avg Goals Conceded: 2.246


In [5]:
get_team_stats('India', df)

Total Matches: 530
Win Rate: 0.358
Avg Goals Scored: 1.374
Avg Goals Conceded: 1.613


In [6]:
get_team_stats('Brazil', df)

Total Matches: 1060
Win Rate: 0.632
Avg Goals Scored: 2.167
Avg Goals Conceded: 0.9


In [7]:
get_team_stats('Argentina', df)

Total Matches: 1067
Win Rate: 0.551
Avg Goals Scored: 1.889
Avg Goals Conceded: 1.006


In [8]:
get_team_stats('France', df)

Total Matches: 936
Win Rate: 0.509
Avg Goals Scored: 1.828
Avg Goals Conceded: 1.288


In [9]:
#build stats for every team in the dataset
teams = pd.unique(df[['home_team', 'away_team']].values.ravel())
print(f"Total unique teams: {len(teams)}")

Total unique teams: 333


In [10]:
# Build stats lookup for all teams
team_stats = {}
for team in teams:
    home_matches = df[df['home_team'] == team]
    away_matches = df[df['away_team'] == team]
    total_matches = len(home_matches) + len(away_matches)

    if total_matches > 0:
        home_wins = len(home_matches[home_matches['result'] == 'home_win'])
        away_wins = len(away_matches[away_matches['result'] == 'away_win'])
        total_wins = home_wins + away_wins
        goals_scored = home_matches['home_score'].sum() + away_matches['away_score'].sum()
        goals_conceded = home_matches['away_score'].sum() + away_matches['home_score'].sum()

        team_stats[team] = {
            'win_rate': total_wins / total_matches,
            'avg_goals_scored': goals_scored / total_matches,
            'avg_goals_conceded': goals_conceded / total_matches
        }

print(f"Stats built for {len(team_stats)} teams")

Stats built for 333 teams


In [12]:
features = []
labels = []

for _, row in df.iterrows():
  home_team = row['home_team']
  away_team = row['away_team']

  if home_team not in team_stats or away_team not in team_stats:
    continue

  home = team_stats[home_team]
  away = team_stats[away_team]

  feature = [
      home['win_rate'] - away['win_rate'],
      home['avg_goals_scored'] - away['avg_goals_scored'],
      home['avg_goals_conceded'] - away['avg_goals_conceded'],
  ]

  features.append(feature)
  labels.append(row['result'])

print(f"Features built for {len(features)} matches")


Features built for 49287 matches


In [16]:
print(f"Features: {features[:5]}")
print(f"Labels: {labels[:10]}")

Features: [[-0.10155217894596424, np.float64(-0.48155200681633326), np.float64(0.2770501714841447)], [0.10155217894596424, np.float64(0.48155200681633326), np.float64(-0.2770501714841447)], [-0.10155217894596424, np.float64(-0.48155200681633326), np.float64(0.2770501714841447)], [0.10155217894596424, np.float64(0.48155200681633326), np.float64(-0.2770501714841447)], [-0.10155217894596424, np.float64(-0.48155200681633326), np.float64(0.2770501714841447)]]
Labels: ['draw', 'home_win', 'home_win', 'draw', 'home_win', 'home_win', 'away_win', 'away_win', 'home_win', 'home_win']


In [18]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

x = np.array(features)
y = np.array(labels)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(x_train, y_train)

predictions = model.predict(x_test)

accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy}")

Accuracy: 0.5293162913369852


In [19]:
import pickle

with open('football_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('team_stats.pkl', 'wb') as f:
    pickle.dump(team_stats,f)

print("Both files saved successfully")

Both files saved successfully
